In [ ]:
from experta import *

class Person(Fact):
    pass

class BMIHealthSystem(KnowledgeEngine):

    @DefFacts()
    def initial_facts(self):
        yield Fact(action='start')

    @Rule(Fact(action='start'), salience=100)
    def start_application(self):
        print("\n" + "="*40)
        print("Welcome to the BMI & Health Expert System")
        print("="*40 + "\n")
        
        try:
            weight = float(input("Enter your Weight (kg): "))
            height = float(input("Enter your Height (m): "))
            bmi = weight / (height ** 2)
            print(f"\nCalculated BMI: {bmi:.2f}")
            
            self.declare(Person(bmi=bmi))
            self.declare(Fact(action='classify_bmi'))
        except ValueError:
            print("Invalid input. Please restart and enter valid numbers.")
            self.declare(Fact(action='start'))

    
    @Rule(Fact(action='classify_bmi'), Person(bmi=P(lambda x: x < 18.5)), salience=90)
    def classify_underweight(self):
        print("\nBMI Category: Underweight")
        self.declare(Person(category='underweight'))
        self.declare(Person(diet='High protein diet for weight Gain'))
        self.declare(Fact(action='ask_activity'))

    @Rule(Fact(action='classify_bmi'), Person(bmi=P(lambda x: 18.5 <= x < 25)), salience=90)
    def classify_normal(self):
        print("\nBMI Category: Normal Weight")
        self.declare(Person(category='normal'))
        self.declare(Person(diet='Balanced diet'))
        self.declare(Fact(action='ask_activity'))

    @Rule(Fact(action='classify_bmi'), Person(bmi=P(lambda x: 25 <= x < 30)), salience=90)
    def classify_overweight(self):
        print("\nBMI Category: Overweight")
        self.declare(Person(category='overweight'))
        self.declare(Person(diet='Low carb Diet'))
        self.declare(Fact(action='ask_activity'))

    @Rule(Fact(action='classify_bmi'), Person(bmi=P(lambda x: x >= 30)), salience=90)
    def classify_obese(self):
        print("\nBMI Category: Obese")
        self.declare(Person(category='obese'))
        self.declare(Person(diet='Ketogenic diet for weight loss'))
        self.declare(Fact(action='ask_activity'))

    @Rule(Fact(action='ask_activity'), salience=80)
    def ask_activity(self):
        print("\n" + "-"*30)
        print("Please determine your activity level:")
        print("- 'high' (>= 3 times/week)")
        print("- 'moderate'")
        print("- 'low'")
        print("- 'sedentary'")
        valid_levels = ['high', 'moderate', 'low', 'sedentary']
        activity = input("Enter your activity level: ").strip().lower()
        while activity not in valid_levels:
            print("Invalid choice. Please enter one of: high, moderate, low, sedentary.")
            activity = input("Enter your activity level: ").strip().lower()
        self.declare(Person(activity=activity))
        self.declare(Fact(action='diagnose'))


    @Rule(Fact(action='diagnose'), Person(category='underweight'), Person(activity=P(lambda x: x in ['high', 'moderate'])))
    def active_underweight(self):
        print("\nDiagnosis: Active Underweight")

    @Rule(Fact(action='diagnose'), Person(category='underweight'), Person(activity=P(lambda x: x in ['low', 'sedentary'])))
    def low_activity_underweight(self):
        print("\nDiagnosis: Low Activity Underweight")


    @Rule(Fact(action='diagnose'), Person(category='normal'), Person(activity='high'))
    def healthy_lifestyle(self):
        print("\nDiagnosis: Healthy Lifestyle")

    @Rule(Fact(action='diagnose'), Person(category='normal'), Person(activity=P(lambda x: x != 'high')))
    def improve_activity(self):
        print("\nDiagnosis: Improve Activity")

    @Rule(Fact(action='diagnose'), Person(category='overweight'), Person(activity='low'))
    def high_risk_overweight(self):
        print("\nDiagnosis: High Risk Overweight")

    @Rule(Fact(action='diagnose'), Person(category='overweight'), Person(activity=P(lambda x: x != 'low')))
    def moderate_risk(self):
        print("\nDiagnosis: Moderate Risk")


    @Rule(Fact(action='diagnose'), Person(category='obese'), Person(activity='sedentary'))
    def severe_risk(self):
        print("\nDiagnosis: Severe Risk")

    @Rule(Fact(action='diagnose'), Person(category='obese'), NOT(Person(activity='sedentary')))
    def obese_but_active(self):
        print("\nDiagnosis: Obese but Active")

    @Rule(Fact(action='diagnose'), Person(diet=MATCH.diet), salience=10)
    def show_diet_recommendation(self, diet):
        print("\n" + "="*40)
        print(f"Recommended Diet: {diet}")
        print("="*40 + "\n")

if __name__ == "__main__":
    while True:
        expert = BMIHealthSystem()
        expert.reset()
        expert.run()
        
        satisfied = input("Are you satisfied? (yes/no): ").lower()
        if satisfied == 'yes':
            print("Thank you for using the system!")
            break
        else:
            print("\nRestarting system...\n")